# SNCP--PPO — Final sistem: v39 (öğrenilmiş risk kafası + Lagrangian PPO)

Bu defter, **v39 öğrenilmiş sistemi** Colab GPU'da uçtan uca üretir. Deploy yolu
**tek bir `SNCPPolicy` forward**'udur; Raspberry Pi'de runtime action shield yoktur.

1. **Kurulum** — repo (v39 dalı) + bağımlılıklar.
2. **(İsteğe bağlı) v34 tabanı** — `checkpoints/sncp_ppo_v34.pt` yoksa Beta politikasını sıfırdan eğit.
3. **v39 eğitimi** — kısa smoke, sonra risk kafası + Lagrangian PPO (v34 warm-start).
4. **Dürüst değerlendirme** — **birincil:** v39, `action_shield=False` (önerilen deploy).
   İsteğe bağlı C0/C1 oracle tavanı: v34 ham / v34+v38 kalkan (öğrenilmeyen tavan, deploy değil).
5. **İstatistik + görseller + indirme**.

Tasarım notu: [`docs/v39-risk-head-lagrangian.md`](docs/v39-risk-head-lagrangian.md).

> v38 eğitimsiz eylem kalkanı **eval-only oracle** olarak durur. Final önerilen sistem
> v39'dur: kısa ufuklu çarpışma kaçınması öğrenmeye gömülür, çıkarımda aday tarama yoktur.

## 0. Çalışma zamanı (GPU)

In [ ]:
!nvidia-smi -L || echo "GPU yok — Çalışma Zamanı > Çalışma zamanı türünü değiştir > T4/A100 GPU seçin."

## 1. Kurulum: repo + bağımlılıklar

PR birleşene kadar **`cursor/v39-risk-head-lagrangian-6377`** dalı zorunludur
(`git fetch` + `git checkout`). Birleştikten sonra `git checkout main` yeter.

In [ ]:
import os
REPO = "sncp-ppo-crowdnav"
V39_BRANCH = "cursor/v39-risk-head-lagrangian-6377"
if not os.path.isdir(REPO):
    !git clone https://github.com/heimdilon/sncp-ppo-crowdnav.git
%cd {REPO}
!git fetch origin
!git checkout {V39_BRANCH}
!git pull --ff-only
!pip -q install -r requirements.txt
print("Kurulum tamam. Dal:", os.popen("git rev-parse --abbrev-ref HEAD").read().strip(), "| CWD:", os.getcwd())

## 2. (İsteğe bağlı) v34 Beta tabanı

v39, `--init_checkpoint checkpoints/sncp_ppo_v34.pt` ile v34-fixed-beta ağırlıklarına
taze risk kafası ekler. Checkpoint zaten varsa **bu hücreyi atlayın**.

**Reçete:** v30 tabanı (ön-MLP + mean+max + yoğunluk müfredatı) + Beta eylem dağılımı,
2.5M adım, robot 1.0 m/s, paper_challenging. A100'de ~3–4 saat.

In [ ]:
!python -m sncp_ppo.train \
  --num_envs 16 --horizon 128 --total_steps 2500000 --lr 1e-4 \
  --fixed_scenario paper_challenging --num_humans 10 --num_humans_range 10 20 \
  --bootstrap_easy_steps 200000 --robot_vpref 1.0 \
  --holdout_scenarios paper_standard paper_challenging --holdout_episodes 50 \
  --pre_mlp --meanmax_pool --action_dist beta --ent_coef 0.001 \
  --save_path checkpoints/sncp_ppo_v34.pt

In [ ]:
# v34 checkpoint'i doğrula (Beta başlığı otomatik algılanır)
import torch, os
assert os.path.exists("checkpoints/sncp_ppo_v34.pt"), "v34 yok — üstteki eğitim hücresini çalıştırın veya checkpoint kopyalayın"
sd = torch.load("checkpoints/sncp_ppo_v34.pt", map_location="cpu", weights_only=True)
print("v34 anahtar sayısı:", len(sd))
print("Beta politikası (actor_logstd YOK):", not any("actor_logstd" in k for k in sd))
print("risk kafası YOK (beklenen):", not any(k.startswith("risk_mlp") or k == "_risk_head" for k in sd))

## 3. v39 smoke (64 adım)

Tam koşudan önce kablolamayı doğrular: risk kafası + Lagrangian bayrakları, v34 init.
Çıktı `checkpoints/sncp_ppo_v39_smoke.pt` — final checkpoint değildir.

In [ ]:
!python -m sncp_ppo.train \
  --risk_head --lagrange_ppo \
  --risk_horizon 6 \
  --risk_bce_coef 1.0 --risk_clearance_coef 0.1 \
  --lagrange_cost_limit 0.05 --lagrange_lr 0.01 \
  --num_envs 2 --horizon 8 --total_steps 64 --eval_freq_updates 0 --lr 1e-4 \
  --fixed_scenario paper_challenging --num_humans 10 \
  --robot_vpref 1.0 \
  --pre_mlp --meanmax_pool --action_dist beta --ent_coef 0.001 \
  --init_checkpoint checkpoints/sncp_ppo_v34.pt \
  --save_path checkpoints/sncp_ppo_v39_smoke.pt

## 4. v39 tam eğitim (öğrenilmiş sistem)

Risk kafası + Lagrangian PPO, v34 warm-start. **Runtime shield yok** — kaçınma
ayrıcalıklı kısa-ufuk etiketleriyle kayba gömülür. A100'de ~3–4 saat; holdout-best
`checkpoints/sncp_ppo_v39.pt`.

In [ ]:
!python -m sncp_ppo.train \
  --risk_head --lagrange_ppo \
  --risk_horizon 6 \
  --risk_bce_coef 1.0 --risk_clearance_coef 0.1 \
  --lagrange_cost_limit 0.05 --lagrange_lr 0.01 \
  --num_envs 16 --horizon 128 --total_steps 2500000 --lr 1e-4 \
  --fixed_scenario paper_challenging --num_humans 10 --num_humans_range 10 20 \
  --bootstrap_easy_steps 200000 --robot_vpref 1.0 \
  --holdout_scenarios paper_standard paper_challenging --holdout_episodes 50 \
  --pre_mlp --meanmax_pool --action_dist beta --ent_coef 0.001 \
  --init_checkpoint checkpoints/sncp_ppo_v34.pt \
  --save_path checkpoints/sncp_ppo_v39.pt

In [ ]:
# v39 checkpoint: risk kafası auto-detect
import torch, os
from sncp_ppo.models import build_policy_for_checkpoint
assert os.path.exists("checkpoints/sncp_ppo_v39.pt"), "v39 yok — tam eğitim hücresini çalıştırın"
sd = torch.load("checkpoints/sncp_ppo_v39.pt", map_location="cpu", weights_only=True)
policy = build_policy_for_checkpoint(sd, robot_vpref=1.0, robot_wmax=1.8)
print("v39 anahtar sayısı:", len(sd))
print("risk_head:", policy.risk_head, "| action_dist:", policy.action_dist)

## 5. Dürüst değerlendirme — v39 shield-off (önerilen deploy)

Aşağıdaki yardımcı **tek bir** `evaluate_density` yolu kullanır.

* **C2 (birincil, önerilen):** `checkpoints/sncp_ppo_v39.pt`, **`action_shield=False`**.
* **C0 / C1 (isteğe bağlı oracle, deploy değil):** aynı v34 checkpoint; kalkan kapalı / açık.
  C1, v38 runtime shield tavanıdır — öğrenilmiş sistemin yerine geçmez.

Protokol: 5 tohum `[100,200,300,400,500]` × 50 bölüm = **250 bölüm/yoğunluk**,
$N=5,10,15,20$, `paper_challenging`, robot 1.0 m/s. Tam C2 tarama ~15–25 dk;
oracle matrisi açıksa ~30–45 dk daha.

In [ ]:
import json, math, time
from sncp_ppo.eval_report import evaluate_density

SEEDS = [100, 200, 300, 400, 500]
DENSITIES = [5, 10, 15, 20]
N_EP = 50

def pooled_se(p, n):
    return math.sqrt(p * (1 - p) / n) if n else float("nan")

def honest_sweep(checkpoint_path, action_shield, out_path, label):
    results = {}
    t0 = time.time()
    for N in DENSITIES:
        blocks, succ, coll, to, steps, isp = [], [], [], [], [], []
        for s in SEEDS:
            eps = evaluate_density(
                checkpoint_path=checkpoint_path, num_humans=N, scenario="paper_challenging",
                n_episodes=N_EP, seed=s, robot_vpref=1.0, human_vpref_override=1.0,
                max_time=None, human_goal_noise=0.0,
                action_shield=action_shield, shield_horizon_steps=6, shield_safety_margin=0.0,
            )
            b = [e.success for e in eps]
            blocks.append(sum(b) / len(b))
            succ += b
            coll += [e.collision for e in eps]
            to += [e.timeout for e in eps]
            steps += [e.steps for e in eps if e.success]
            isp += [e.avg_i_sp for e in eps]
            print(f"  [{label}] N={N} seed={s} succ={blocks[-1]*100:4.1f}%  ({time.time()-t0:.0f}s)", flush=True)
        n = len(succ); p = sum(succ) / n
        results[str(N)] = {
            "block_means": blocks, "pooled_success": p, "pooled_se": pooled_se(p, n),
            "pooled_collision": sum(coll) / n, "pooled_timeout": sum(to) / n,
            "avg_success_steps": (sum(steps) / len(steps)) if steps else float("nan"),
            "avg_i_sp": sum(isp) / len(isp), "n": n,
        }
        r = results[str(N)]
        print(f"== [{label}] N={N} POOLED succ={p*100:.1f}%  coll={r['pooled_collision']*100:.1f}%  to={r['pooled_timeout']*100:.1f}% ==", flush=True)
        json.dump(results, open(out_path, "w"), indent=2)
    return results

# C2 — önerilen deploy: öğrenilmiş v39, kalkan KAPALI
v39 = honest_sweep(
    "checkpoints/sncp_ppo_v39.pt", action_shield=False,
    out_path="v39_multiseed_result.json", label="C2 v39 shield-off",
)

# C0/C1 — v38 runtime shield oracle tavanı. Deploy yolu DEĞİL.
RUN_ORACLE_MATRIX = False
if RUN_ORACLE_MATRIX:
    v34 = honest_sweep(
        "checkpoints/sncp_ppo_v34.pt", action_shield=False,
        out_path="v34_multiseed_result.json", label="C0 v34 ham",
    )
    v38 = honest_sweep(
        "checkpoints/sncp_ppo_v34.pt", action_shield=True,
        out_path="v38_multiseed_result.json", label="C1 v34+v38 kalkan",
    )
print("\nBİRİNCİL TARAMA TAMAM -> v39_multiseed_result.json (action_shield=False)")

## 6. İstatistiksel analiz — v39 (birincil)

Wilson %95 GA. Oracle JSON'ları varsa C2 vs C0 (öğrenme kazancı) ve C1 vs C0
(shield tavanı) ayrıca basılır. Bonferroni eşiği $\alpha = 0.05/4 = 0.0125$.

In [ ]:
import json, math, os
from scipy.stats import norm

Z = 1.959963984540054
DENS = [5, 10, 15, 20]
ALPHA = 0.0125
v39 = json.load(open("v39_multiseed_result.json"))

def wilson(k, n):
    p = k / n; d = 1 + Z*Z/n; c = (p + Z*Z/(2*n)) / d
    h = Z*math.sqrt(p*(1-p)/n + Z*Z/(4*n*n)) / d
    return (c-h)*100, (c+h)*100

def ztest(k1, n1, k2, n2):
    p1, p2 = k1/n1, k2/n2; pp = (k1+k2)/(n1+n2)
    se = math.sqrt(pp*(1-pp)*(1/n1+1/n2)); z = (p1-p2)/se if se > 0 else 0.0
    return z, 2*norm.sf(abs(z))

def cohen_h(p1, p2):
    return 2*(math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

print("C2 v39 shield-off (önerilen deploy)")
print(f"{'N':>3} | {'succ [95% GA]':>22} | {'coll':>8} | {'timeout':>8}")
print("-" * 52)
for N in DENS:
    s = str(N); n = v39[s]["n"]
    ks = round(v39[s]["pooled_success"]*n)
    lo, hi = wilson(ks, n)
    print(f"{N:>3} | {v39[s]['pooled_success']*100:>6.1f} [{lo:>5.1f},{hi:>5.1f}] | {v39[s]['pooled_collision']*100:>7.1f} | {v39[s]['pooled_timeout']*100:>7.1f}")

if os.path.exists("v34_multiseed_result.json") and os.path.exists("v38_multiseed_result.json"):
    v34 = json.load(open("v34_multiseed_result.json"))
    v38 = json.load(open("v38_multiseed_result.json"))
    print("\nİsteğe bağlı oracle: C1 (v38 kalkan) vs C0 (v34 ham) — deploy değil")
    print(f"{'N':>3} | {'v34 succ':>8} | {'v38 succ [95% GA]':>22} | {'dPP succ (p)':>16} | {'v34 coll':>8} | {'v38 coll':>8} | {'dPP coll (p)':>16}")
    print("-" * 104)
    for N in DENS:
        s = str(N); n = v38[s]["n"]
        ks, kb = round(v38[s]["pooled_success"]*n), round(v34[s]["pooled_success"]*n)
        lo, hi = wilson(ks, n); zs, ps = ztest(ks, n, kb, n)
        cc, cb = round(v38[s]["pooled_collision"]*n), round(v34[s]["pooled_collision"]*n)
        zc, pc = ztest(cc, n, cb, n)
        ds = (v38[s]["pooled_success"]-v34[s]["pooled_success"])*100
        dc = (v38[s]["pooled_collision"]-v34[s]["pooled_collision"])*100
        gs = "*" if ps < ALPHA else " "; gc = "*" if pc < ALPHA else " "
        print(f"{N:>3} | {v34[s]['pooled_success']*100:>7.1f} | {v38[s]['pooled_success']*100:>6.1f} [{lo:>5.1f},{hi:>5.1f}] | {ds:>+6.1f} ({ps:6.4f}){gs} | {v34[s]['pooled_collision']*100:>7.1f} | {v38[s]['pooled_collision']*100:>7.1f} | {dc:>+6.1f} ({pc:6.4f}){gc}")
    print("\n*  = Bonferroni-anlamlı (p < 0.0125).")
    print("Cohen h (N=20 başarı, C1 vs C0):", round(cohen_h(v38['20']['pooled_success'], v34['20']['pooled_success']), 3))
else:
    print("\nOracle JSON yok (RUN_ORACLE_MATRIX=False). C2 tablosu yeterli; C0/C1 için tarama hücresinde bayrağı açın.")

## 7. Görseller: v39 shield-off (deploy) + isteğe bağlı C0/C1 tavanı

In [ ]:
import json, os, numpy as np, matplotlib.pyplot as plt
v39 = json.load(open("v39_multiseed_result.json"))
N = np.array([5, 10, 15, 20])
s39 = np.array([v39[str(n)]["pooled_success"]*100 for n in N])
c39 = np.array([v39[str(n)]["pooled_collision"]*100 for n in N])
def se(a):
    p = a/100.0; return 1.96*np.sqrt(p*(1-p)/250.0)*100.0

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.errorbar(N, s39, yerr=se(s39), fmt="-o", color="#1F4E79", lw=2.6, ms=8, capsize=3, label="C2 v39 (shield-off)")
a2.errorbar(N, c39, yerr=se(c39), fmt="-o", color="#1F4E79", lw=2.6, ms=8, capsize=3, label="C2 v39 (shield-off)")
if os.path.exists("v34_multiseed_result.json") and os.path.exists("v38_multiseed_result.json"):
    v34 = json.load(open("v34_multiseed_result.json"))
    v38 = json.load(open("v38_multiseed_result.json"))
    s34 = np.array([v34[str(n)]["pooled_success"]*100 for n in N])
    c34 = np.array([v34[str(n)]["pooled_collision"]*100 for n in N])
    s38 = np.array([v38[str(n)]["pooled_success"]*100 for n in N])
    c38 = np.array([v38[str(n)]["pooled_collision"]*100 for n in N])
    a1.plot(N, s34, "--s", color="#BA7517", lw=2, ms=7, label="C0 v34 ham")
    a1.plot(N, s38, ":D", color="#6B6B6B", lw=2, ms=7, label="C1 v38 kalkan (oracle, deploy değil)")
    a2.plot(N, c34, "--s", color="#BA7517", lw=2, ms=7, label="C0 v34")
    a2.plot(N, c38, ":D", color="#6B6B6B", lw=2, ms=7, label="C1 v38 kalkan")
a1.axhline(94, color="gray", ls=":", label="Makale ~%94")
a1.set_title("Başarı"); a1.set_xlabel("Yaya sayısı N"); a1.set_ylabel("Başarı (%)"); a1.set_xticks(N); a1.set_ylim(80, 101); a1.legend(); a1.grid(alpha=.3)
a2.set_title("Çarpışma"); a2.set_xlabel("Yaya sayısı N"); a2.set_ylabel("Çarpışma (%)"); a2.set_xticks(N); a2.set_ylim(-1, 25); a2.legend(); a2.grid(alpha=.3)
fig.tight_layout(); fig.savefig("v39_final_comparison.png", dpi=150); plt.show()

In [ ]:
# v39 yörüngeler (N=10 ve N=20) — kalkan KAPALI; öğrenilmiş politika tek forward
from sncp_ppo.eval_report import render_trajectory
from IPython.display import Image, display
for n in (10, 20):
    render_trajectory(
        checkpoint_path="checkpoints/sncp_ppo_v39.pt", output_path=f"v39_traj_n{n}.png",
        num_humans=n, scenario="paper_challenging", seed=100,
        robot_vpref=1.0, human_vpref_override=1.0, max_time=50.0, action_shield=False,
    )
    display(Image(f"v39_traj_n{n}.png"))

## 8. Sonuç ve yorum

* **v39** kısa ufuklu çarpışma riskini risk kafası + Lagrangian PPO ile **öğrenir**.
  Deploy / Pi yolu kalkan döngüsü içermez (`action_shield=False`).
* **v34** hâlâ warm-start tabanıdır; kendi başına final deploy önerisi değildir.
* **v38 kalkan** isteğe bağlı **oracle tavanıdır** (C1). Öğrenilmeyen runtime süzgeç;
  önerilen final sistem değildir.
* Planlanan matris: C0 v34 raw vs C1 v34+v38 shield vs C2 v39 shield-off.
  Ayrıntı: `docs/v39-risk-head-lagrangian.md`.

**Üretilen artefaktlar:** `checkpoints/sncp_ppo_v39.pt`, `v39_multiseed_result.json`,
`v39_final_comparison.png`, `v39_traj_n10.png`, `v39_traj_n20.png`
(ve varsa v34 checkpoint + C0/C1 JSON).

## 9. Artefaktları indir (Colab)

v39 checkpoint + shield-off sweep JSON + figürler tek ZIP'te. Oracle dosyaları
varsa pakete eklenir.

In [ ]:
import os, zipfile
DOWNLOAD = True
bundle = "sncp_ppo_v39_final_artifacts.zip"
artifacts = [
    "v39_multiseed_result.json",
    "v34_multiseed_result.json", "v38_multiseed_result.json",
    "v39_final_comparison.png", "v39_traj_n10.png", "v39_traj_n20.png",
    "checkpoints/sncp_ppo_v39.pt", "checkpoints/sncp_ppo_v34.pt",
]
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in artifacts:
        if os.path.exists(f):
            z.write(f); print("  +", f)
print("paket:", bundle)
if DOWNLOAD:
    try:
        from google.colab import files
        files.download(bundle)
    except Exception as e:
        print("indirme atlandı (Colab dışı ortam):", e)